# Tong hop bieu do cho bao cao - Module Nhan dien

Notebook GOP 3 phan (truoc day la 3 file rieng):
- **PHAN 1 - EDA** (Buoi 2): kham pha du lieu CASIA-WebFace - 6 bieu do
- **PHAN 2 - Danh gia** (Buoi 5): metrics + 3 bieu do (score dist, ROC, FAR/FRR)
- **PHAN 3 - Grad-CAM** (Buoi 5): truc quan vung model tap trung - 2 bieu do

**Setup dung chung 1 lan** (tai CASIA, load model) -> chay tuan tu -> tai TAT CA hinh ve 1 lan.

**Chay tren Colab CPU** (GPU dang bi chan). EDA + Grad-CAM nhanh; phan Eval cham hon
(trich embedding tren CPU) - co the giam so anh neu can.

---
### Truoc khi chay:
- Runtime -> Change runtime type -> **None (CPU)** -> Save


# PHAN 0 - SETUP DUNG CHUNG (chay 1 lan)

Mount Drive, tai CASIA, khoi phuc code, load model. Cac phan sau dung lai ket qua nay.

### 0.1 - Duong dan + Drive

In [ ]:
import os
DATA_DIR = '/content/casia-webface'
CKPT = '/content/drive/MyDrive/attendance/checkpoints/recognition/last.pt'

if CKPT.startswith('/content/drive') and not os.path.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

# Thu muc luu TAT CA hinh
ALL_FIG = '/content/all_figures'
os.makedirs(ALL_FIG, exist_ok=True)
print("DATA_DIR:", os.path.exists(DATA_DIR), "| CKPT:", os.path.exists(CKPT))

### 0.2 - Tai CASIA (neu mat) + khoi phuc code recognition
Dien Kaggle key neu CASIA chua co.

In [ ]:
import os, zipfile
if not os.path.exists(DATA_DIR):
    os.environ['KAGGLE_USERNAME'] = 'username_cua_ban'
    os.environ['KAGGLE_KEY'] = 'key_cua_ban'
    !pip install -q kaggle
    !kaggle datasets download -d ntl0601/casia-webface -p /content --unzip

DRIVE_CODE = '/content/drive/MyDrive/attendance/recognition_colab.zip'
os.makedirs('/content/recognition', exist_ok=True)
if os.path.exists(DRIVE_CODE):
    with zipfile.ZipFile(DRIVE_CODE) as z:
        z.extractall('/content/recognition')
    print("Khoi phuc code tu Drive")
else:
    from google.colab import files
    print("Upload recognition_colab.zip...")
    up = files.upload()
    for fn in up:
        if fn.endswith('.zip'):
            with zipfile.ZipFile(fn) as z:
                z.extractall('/content/recognition')

import sys
sys.path.insert(0, '/content/recognition')
!pip install -q scikit-learn opencv-python-headless pytorch_lightning
print("Code san sang")

### 0.3 - Sua nhan checkpoint (phong loi metadata) + Load model 1 lan
Tu kiem tra weights co SE khong, sua nhan cho khop, roi load model dung cho ca Eval va Grad-CAM.

In [ ]:
import torch

# Sua nhan backbone_name cho khop weights that (idempotent)
ck = torch.load(CKPT, map_location='cpu', weights_only=False)
has_se = any('se_block' in k for k in ck['backbone_state'].keys())
correct = 'ir_50_se' if has_se else 'ir_50'
if ck.get('backbone_name') != correct:
    ck['backbone_name'] = correct
    torch.save(ck, CKPT)
    print(f"Da sua nhan -> {correct}")
else:
    print(f"Nhan OK: {correct}")

from extract_embedding import EmbeddingExtractor, preprocess_bgr
device = 'cpu'
embedder = EmbeddingExtractor(CKPT, device=device)
model = embedder.model
model.eval()
print("Backbone:", embedder.backbone_name, "| Model loaded")

# PHAN 1 - EDA (Buoi 2)

Kham pha du lieu CASIA-WebFace. Phan nay chi DOC anh, khong dung model -> rat nhanh.

### 1.1 - Thong ke tong quan
Bien: so identity, tong anh, anh/nguoi. Dung de: biet do kho bai toan + du data khong.

In [ ]:
import numpy as np
ids = [d for d in sorted(os.listdir(DATA_DIR)) if os.path.isdir(os.path.join(DATA_DIR, d))]
counts = np.array([len([f for f in os.listdir(os.path.join(DATA_DIR, p))
                        if f.lower().endswith(('.jpg','.jpeg','.png'))]) for p in ids])
print(f"So nguoi: {len(ids):,} | Tong anh: {counts.sum():,}")
print(f"Anh/nguoi: TB={counts.mean():.1f} | trung vi={np.median(counts):.0f} | "
      f"min={counts.min()} | max={counts.max()}")

### 1.2 - Phan bo so anh/nguoi (class balance)
Truc X = so anh/nguoi, Y = so nguoi. Dung de: danh gia mat can bang lop.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(counts, bins=40, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].axvline(counts.mean(), color='red', linestyle='--', label=f'TB={counts.mean():.1f}')
axes[0].axvline(np.median(counts), color='green', linestyle='--', label=f'Trung vi={np.median(counts):.0f}')
axes[0].set_xlabel('So anh moi nguoi'); axes[0].set_ylabel('So nguoi')
axes[0].set_title('Phan bo so anh / nguoi'); axes[0].legend()
labels=['1-5','6-10','11-20','21-50','51-100','100+']
binned=np.histogram(counts, bins=[0,5,10,20,50,100,99999])[0]
axes[1].bar(labels, binned, color='coral', edgecolor='black')
axes[1].set_xlabel('Nhom so anh'); axes[1].set_ylabel('So nguoi'); axes[1].set_title('So nguoi theo nhom')
for i,v in enumerate(binned): axes[1].text(i, v, str(v), ha='center', va='bottom')
plt.tight_layout(); plt.savefig(f'{ALL_FIG}/eda_01_class_distribution.png', dpi=120, bbox_inches='tight'); plt.show()

### 1.3 - Anh mau (sanity check chat luong + alignment)

In [ ]:
import cv2, random
random.seed(42)
sample_ids = random.sample(ids, min(20, len(ids)))
fig, axes = plt.subplots(4, 5, figsize=(14, 11)); axes = axes.flatten()
for ax, pid in zip(axes, sample_ids):
    pdir = os.path.join(DATA_DIR, pid)
    imgs = [f for f in os.listdir(pdir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    if imgs:
        img = cv2.cvtColor(cv2.imread(os.path.join(pdir, imgs[0])), cv2.COLOR_BGR2RGB)
        ax.imshow(img); ax.set_title(f'ID {pid}\n{img.shape[1]}x{img.shape[0]}', fontsize=8)
    ax.axis('off')
for ax in axes[len(sample_ids):]: ax.axis('off')
plt.suptitle('Anh mau tu 20 nguoi', fontsize=14); plt.tight_layout()
plt.savefig(f'{ALL_FIG}/eda_02_sample_faces.png', dpi=120, bbox_inches='tight'); plt.show()

### 1.4 - Bien thien trong cung 1 nguoi (intra-class)

In [ ]:
idx = int(np.argmax(counts)); pid = ids[idx]
pdir = os.path.join(DATA_DIR, pid)
imgs = sorted([f for f in os.listdir(pdir) if f.lower().endswith(('.jpg','.jpeg','.png'))])[:10]
fig, axes = plt.subplots(2, 5, figsize=(14, 6)); axes = axes.flatten()
for ax, fn in zip(axes, imgs):
    ax.imshow(cv2.cvtColor(cv2.imread(os.path.join(pdir, fn)), cv2.COLOR_BGR2RGB)); ax.axis('off')
for ax in axes[len(imgs):]: ax.axis('off')
plt.suptitle(f'10 anh cung 1 nguoi (ID {pid})', fontsize=13); plt.tight_layout()
plt.savefig(f'{ALL_FIG}/eda_03_intra_class.png', dpi=120, bbox_inches='tight'); plt.show()

### 1.5 - Do sang (lighting) + Mau RGB
Do sang: X=do sang 0-255, Y=so anh. Mau: X=gia tri pixel, Y=tan suat, 3 duong R/G/B.

In [ ]:
sample_paths=[]
for pid in random.sample(ids, min(200,len(ids))):
    pdir=os.path.join(DATA_DIR,pid)
    fs=[f for f in os.listdir(pdir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    if fs: sample_paths.append(os.path.join(pdir, random.choice(fs)))

brightness=[]; hist_r=np.zeros(256); hist_g=np.zeros(256); hist_b=np.zeros(256)
for p in sample_paths[:400]:
    img=cv2.imread(p)
    if img is None: continue
    brightness.append(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY).mean())
    rgb=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    hist_r+=cv2.calcHist([rgb],[0],None,[256],[0,256]).flatten()
    hist_g+=cv2.calcHist([rgb],[1],None,[256],[0,256]).flatten()
    hist_b+=cv2.calcHist([rgb],[2],None,[256],[0,256]).flatten()
brightness=np.array(brightness)

fig, ax = plt.subplots(1,2, figsize=(15,5))
ax[0].hist(brightness, bins=40, color='goldenrod', edgecolor='black', alpha=0.8)
ax[0].axvline(brightness.mean(), color='red', linestyle='--', label=f'TB={brightness.mean():.0f}')
ax[0].set_xlabel('Do sang TB (0-255)'); ax[0].set_ylabel('So anh'); ax[0].set_title('Phan bo do sang'); ax[0].legend()
ax[1].plot(hist_r,color='red',label='R',alpha=0.7); ax[1].plot(hist_g,color='green',label='G',alpha=0.7)
ax[1].plot(hist_b,color='blue',label='B',alpha=0.7)
ax[1].set_xlabel('Gia tri pixel'); ax[1].set_ylabel('Tan suat'); ax[1].set_title('Phan bo mau RGB'); ax[1].legend()
plt.tight_layout(); plt.savefig(f'{ALL_FIG}/eda_04_brightness_color.png', dpi=120, bbox_inches='tight'); plt.show()
print(f"Do sang TB: {brightness.mean():.0f}/255")

# PHAN 2 - DANH GIA (Buoi 5)

Danh gia verification: tao cap cung nguoi / khac nguoi, do tuong dong cosine.

**Luu y CPU:** trich embedding cham. Mac dinh 80 nguoi x 8 anh (~640 anh, ~5-8 phut CPU).
Neu cham qua, giam N_IDENTITIES.

### 2.1 - Trich embedding (dung model da load o PHAN 0)

In [ ]:
N_IDENTITIES = 80     # giam neu CPU cham (vd 50)
MAX_IMG_PER_ID = 8

random.seed(42)
ids_shuf = ids[:]; random.shuffle(ids_shuf)
emb_by_id = {}
for pid in ids_shuf:
    pdir = os.path.join(DATA_DIR, pid)
    imgs = [f for f in os.listdir(pdir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    if len(imgs) < 4: continue
    imgs = random.sample(imgs, min(MAX_IMG_PER_ID, len(imgs)))
    embs=[]
    for fn in imgs:
        img=cv2.imread(os.path.join(pdir,fn))
        if img is not None: embs.append(embedder.embed_bgr(img))
    if len(embs)>=2: emb_by_id[pid]=np.stack(embs)
    if len(emb_by_id)>=N_IDENTITIES: break
print(f"Da trich {len(emb_by_id)} nguoi, {sum(len(v) for v in emb_by_id.values())} anh")

### 2.2 - Tao cap so sanh

In [ ]:
same_sims, diff_sims = [], []
id_list = list(emb_by_id.keys())
for pid, embs in emb_by_id.items():
    n=len(embs)
    for i in range(n):
        for j in range(i+1,n): same_sims.append(float(np.dot(embs[i],embs[j])))
target=len(same_sims); att=0
while len(diff_sims)<target and att<target*5:
    a,b=random.sample(id_list,2)
    ea=emb_by_id[a][random.randint(0,len(emb_by_id[a])-1)]
    eb=emb_by_id[b][random.randint(0,len(emb_by_id[b])-1)]
    diff_sims.append(float(np.dot(ea,eb))); att+=1
same_sims=np.array(same_sims); diff_sims=np.array(diff_sims)
print(f"Same: {len(same_sims):,} | Diff: {len(diff_sims):,}")
print(f"Cosine cung nguoi TB={same_sims.mean():.3f} | khac nguoi TB={diff_sims.mean():.3f}")

### 2.3 - Score Distribution (bieu do chu luc)
X=cosine, Y=mat do, xanh=cung nguoi, do=khac nguoi. 2 phan bo tach roi = model tot.

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
ax.hist(diff_sims, bins=60, alpha=0.6, color='crimson', label=f'Khac nguoi (TB={diff_sims.mean():.2f})', density=True)
ax.hist(same_sims, bins=60, alpha=0.6, color='steelblue', label=f'Cung nguoi (TB={same_sims.mean():.2f})', density=True)
ax.axvline(0.30, color='black', linestyle='--', linewidth=2, label='Nguong=0.30')
ax.set_xlabel('Cosine similarity'); ax.set_ylabel('Mat do')
ax.set_title('Phan bo do tuong dong: Cung nguoi vs Khac nguoi'); ax.legend()
plt.tight_layout(); plt.savefig(f'{ALL_FIG}/eval_01_score_distribution.png', dpi=130, bbox_inches='tight'); plt.show()

### 2.4 - ROC + AUC
X=FAR, Y=TAR. AUC gan 1 = tot.

In [ ]:
from sklearn.metrics import roc_curve, auc
labels=np.concatenate([np.ones_like(same_sims), np.zeros_like(diff_sims)])
scores=np.concatenate([same_sims, diff_sims])
fpr,tpr,_=roc_curve(labels,scores); roc_auc=auc(fpr,tpr)
fig,ax=plt.subplots(figsize=(7,7))
ax.plot(fpr,tpr,color='darkorange',lw=2.5,label=f'ROC (AUC={roc_auc:.4f})')
ax.plot([0,1],[0,1],color='navy',lw=1.5,linestyle='--',label='Random')
ax.set_xlabel('FAR'); ax.set_ylabel('TAR'); ax.set_title('ROC Curve'); ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{ALL_FIG}/eval_02_roc_curve.png', dpi=130, bbox_inches='tight'); plt.show()
print(f"AUC={roc_auc:.4f}")

### 2.5 - FAR/FRR + EER
X=nguong, Y=ti le loi. Do=FAR, xanh=FRR, EER=diem FAR=FRR.

In [ ]:
thr=np.linspace(-0.2,1.0,200)
far=np.array([np.mean(diff_sims>=t) for t in thr])
frr=np.array([np.mean(same_sims<t) for t in thr])
ei=np.argmin(np.abs(far-frr)); eer=(far[ei]+frr[ei])/2; eer_t=thr[ei]
fig,ax=plt.subplots(figsize=(10,6))
ax.plot(thr,far,color='crimson',lw=2,label='FAR (nhan nham nguoi la)')
ax.plot(thr,frr,color='steelblue',lw=2,label='FRR (tu choi nguoi that)')
ax.axvline(eer_t,color='green',linestyle='--',label=f'EER={eer:.3f} @ {eer_t:.2f}')
ax.axvline(0.30,color='black',linestyle=':',label='Nguong dang dung=0.30')
ax.set_xlabel('Threshold'); ax.set_ylabel('Error Rate'); ax.set_title('FAR / FRR theo Threshold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{ALL_FIG}/eval_03_far_frr.png', dpi=130, bbox_inches='tight'); plt.show()

def acc_at(t):
    tp=np.sum(same_sims>=t); fn=np.sum(same_sims<t); tn=np.sum(diff_sims<t); fp=np.sum(diff_sims>=t)
    return (tp+tn)/(tp+tn+fp+fn)
print("="*45)
print(f"  AUC={roc_auc:.4f} | EER={eer:.4f}")
print(f"  Accuracy @0.30={acc_at(0.30):.4f} | @EER={acc_at(eer_t):.4f}")
print(f"  Cosine cung/khac nguoi: {same_sims.mean():.3f} / {diff_sims.mean():.3f}")
print("="*45)

# PHAN 3 - GRAD-CAM (Buoi 5)

Truc quan vung khuon mat model tap trung. Dung model da load (bat gradient).

### 3.1 - Bat gradient + cai dat Grad-CAM

In [ ]:
import torch.nn.functional as F
for p in model.parameters(): p.requires_grad_(True)

_act={}; _grad={}
def fwd_hook(m,i,o): _act['v']=o
def bwd_hook(m,gi,go): _grad['v']=go[0]
target_layer = model.body[-1]
target_layer.register_forward_hook(fwd_hook)
target_layer.register_full_backward_hook(bwd_hook)

def compute_gradcam(img_bgr):
    x=preprocess_bgr(img_bgr).to(device); x.requires_grad_(True)
    model.zero_grad()
    normalized, norm = model(x)
    norm.sum().backward()
    acts=_act['v'][0]; grads=_grad['v'][0]
    w=grads.mean(dim=(1,2))
    cam=F.relu((w[:,None,None]*acts).sum(0)).detach().cpu().numpy()
    cam=cam-cam.min(); cam=cam/(cam.max()+1e-9)
    return cv2.resize(cam,(112,112))
print("Grad-CAM san sang")

### 3.2 - Grad-CAM 6 nguoi (goc | heatmap | overlay)
Vung do/vang = model tap trung. Mong doi: trung tam mat (mat, mui, long may).

In [ ]:
random.seed(0)
sample_ids=random.sample(ids,6)
fig,axes=plt.subplots(6,3,figsize=(9,18))
for r,pid in enumerate(sample_ids):
    pdir=os.path.join(DATA_DIR,pid)
    fn=[f for f in os.listdir(pdir) if f.lower().endswith(('.jpg','.jpeg','.png'))][0]
    img=cv2.imread(os.path.join(pdir,fn)); img_rgb=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    if img.shape[:2]!=(112,112): img=cv2.resize(img,(112,112)); img_rgb=cv2.resize(img_rgb,(112,112))
    cam=compute_gradcam(img)
    hm=cv2.cvtColor(cv2.applyColorMap(np.uint8(255*cam),cv2.COLORMAP_JET),cv2.COLOR_BGR2RGB)
    ov=np.uint8(0.5*img_rgb+0.5*hm)
    axes[r,0].imshow(img_rgb); axes[r,0].set_title(f'Goc (ID {pid})',fontsize=10)
    axes[r,1].imshow(cam,cmap='jet'); axes[r,1].set_title('Heatmap',fontsize=10)
    axes[r,2].imshow(ov); axes[r,2].set_title('Overlay',fontsize=10)
    for c in range(3): axes[r,c].axis('off')
plt.suptitle('Grad-CAM: vung model tap trung khi nhan dien',fontsize=14,y=1.0)
plt.tight_layout(); plt.savefig(f'{ALL_FIG}/gradcam_01_grid.png',dpi=120,bbox_inches='tight'); plt.show()

### 3.3 - Grad-CAM nhat quan 1 nguoi (bat bien tu the)

In [ ]:
pid=max(ids,key=lambda p:len(os.listdir(os.path.join(DATA_DIR,p))))
pdir=os.path.join(DATA_DIR,pid)
imgs=sorted([f for f in os.listdir(pdir) if f.lower().endswith(('.jpg','.jpeg','.png'))])[:5]
fig,axes=plt.subplots(2,5,figsize=(15,6))
for c,fn in enumerate(imgs):
    img=cv2.imread(os.path.join(pdir,fn)); img_rgb=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    if img.shape[:2]!=(112,112): img=cv2.resize(img,(112,112)); img_rgb=cv2.resize(img_rgb,(112,112))
    cam=compute_gradcam(img)
    hm=cv2.cvtColor(cv2.applyColorMap(np.uint8(255*cam),cv2.COLORMAP_JET),cv2.COLOR_BGR2RGB)
    ov=np.uint8(0.5*img_rgb+0.5*hm)
    axes[0,c].imshow(img_rgb); axes[0,c].axis('off')
    axes[1,c].imshow(ov); axes[1,c].axis('off')
plt.suptitle(f'Grad-CAM nhat quan 1 nguoi (ID {pid})',fontsize=13)
plt.tight_layout(); plt.savefig(f'{ALL_FIG}/gradcam_02_consistency.png',dpi=120,bbox_inches='tight'); plt.show()

# PHAN CUOI - Tai TAT CA hinh ve 1 lan

Gom tat ca bieu do (EDA + Eval + Grad-CAM) vao 1 file zip.

In [ ]:
import shutil
print("Cac hinh da tao:")
for f in sorted(os.listdir(ALL_FIG)): print("  ", f)
shutil.make_archive('/content/all_charts', 'zip', ALL_FIG)
from google.colab import files
files.download('/content/all_charts.zip')
print("\nDa tai all_charts.zip")

## Xong!

1 file `all_charts.zip` chua tat ca bieu do cho bao cao:
- eda_01..04: EDA (Buoi 2)
- eval_01..03: Danh gia (Buoi 5)
- gradcam_01..02: Grad-CAM (Buoi 5)

So lieu la cua model BAN -> ghi dung so chay ra vao bao cao.
